
# MTA Daily Ridership — Week 2 Exploratory Data Analysis

## Project Introduction and Objectives

Week 1 built and validated a daily star schema from the MTA Daily Ridership dataset. This notebook performs Week 2 exploratory analysis and answers 12 business questions across system performance, time-based behaviour, COVID-19 impact and recovery, and operational relationships.

Guiding rules:

- The Week 1 cleaned CSV is the only analysis input; no source values are changed.
- Transportation systems have different units. Raw activity is never summed across systems, and cross-system comparisons use unit-free measures (recovery percentage, indices, ratios, correlations).
- Zero activity days are retained. They are not treated as missing.
- Recovery percentages above 100 are retained and investigated; they are not removed as outliers.
- The February 2023 source methodology change affected the **pre-pandemic comparison percentages**, not the activity estimates. Recovery trends are interpreted with that caveat.
- Correlation is association only, not causation.



## 2. Imports and Configuration

All paths are resolved from the repository root. The reusable helpers live in `WEEK TWO/python/analysis_utils.py`, which reads only `WEEK ONE/MTA_Daily_Ridership_Cleaned.csv`.


In [1]:

import os
import sys
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def locate_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "WEEK ONE" / "python").is_dir() and (candidate / "WEEK TWO" / "python").is_dir():
            return candidate
    raise FileNotFoundError("Run from inside the repository or set MTA_PROJECT_ROOT.")


_env_root = os.environ.get("MTA_PROJECT_ROOT")
PROJECT_ROOT = locate_root(Path(_env_root) if _env_root else Path.cwd()).resolve()
sys.path.insert(0, str(PROJECT_ROOT / "WEEK TWO" / "python"))

import analysis_utils as au

au.configure_plotting()
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

VIZ_DIR = au.VIZ_DIR
REPORT_DIR = PROJECT_ROOT / "WEEK TWO" / "reports"
VIZ_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)


def save_fig(fig, filename):
    fig.savefig(VIZ_DIR / filename, bbox_inches="tight")
    plt.close(fig)


def fmt_date(value):
    return "not reached" if pd.isna(value) else pd.Timestamp(value).strftime("%Y-%m-%d")


print(f"Project root: {PROJECT_ROOT.name}")
print(f"Cleaned input: {au.CLEANED_PATH.relative_to(PROJECT_ROOT).as_posix()}")
print(f"Visualizations: {VIZ_DIR.relative_to(PROJECT_ROOT).as_posix()}")


Project root: MTA Project
Cleaned input: WEEK ONE/MTA_Daily_Ridership_Cleaned.csv
Visualizations: WEEK TWO/visualizations



## 3. Data Loading

The wide table keeps the 15 published columns. The tidy table has one row per date and transportation system (1,706 x 7 = 11,942 rows) and carries the verified unit for each system.


In [2]:

wide = au.load_cleaned()
long = au.build_long(wide)

print(f"Wide shape: {wide.shape}")
print(f"Tidy shape: {long.shape}")
print(f"Systems: {', '.join(au.SYSTEM_NAMES)}")
display(wide.head(3))
display(long.head(7))


Wide shape: (1706, 15)
Tidy shape: (11942, 8)
Systems: Subways, Buses, LIRR, Metro-North, Access-A-Ride, Bridges and Tunnels, Staten Island Railway


,Date,Subways: Total Estimated Ridership,Subways: % of Comparable Pre-Pandemic Day,Buses: Total Estimated Ridership,Buses: % of Comparable Pre-Pandemic Day,LIRR: Total Estimated Ridership,LIRR: % of Comparable Pre-Pandemic Day,Metro-North: Total Estimated Ridership,Metro-North: % of Comparable Pre-Pandemic Day,Access-A-Ride: Total Scheduled Trips,Access-A-Ride: % of Comparable Pre-Pandemic Day,Bridges and Tunnels: Total Traffic,Bridges and Tunnels: % of Comparable Pre-Pandemic Day,Staten Island Railway: Total Estimated Ridership,Staten Island Railway: % of Comparable Pre-Pandemic Day
0,2020-03-01,2212965,97,984908,99,86790,100,55825,59,19922,113,786960,98,1636,52
1,2020-03-02,5329915,96,2209066,99,321569,103,180701,66,30338,102,874619,95,17140,107
2,2020-03-03,5481103,98,2228608,99,319727,102,190648,69,32767,110,882175,96,17453,109


,date,transport_id,transport,activity,recovery_pct,measure,unit,unit_family
0,2020-03-01,1,Subways,2212965,97,Estimated Ridership,estimated riders,ridership
1,2020-03-02,1,Subways,5329915,96,Estimated Ridership,estimated riders,ridership
2,2020-03-03,1,Subways,5481103,98,Estimated Ridership,estimated riders,ridership
3,2020-03-04,1,Subways,5498809,99,Estimated Ridership,estimated riders,ridership
4,2020-03-05,1,Subways,5496453,99,Estimated Ridership,estimated riders,ridership
5,2020-03-06,1,Subways,5189447,93,Estimated Ridership,estimated riders,ridership
6,2020-03-07,1,Subways,2814637,92,Estimated Ridership,estimated riders,ridership



## 4. Data Quality Review

Checks cover shape, dtypes, missing values, duplicates, date continuity, negative values, and retained zero activity. A failure here would stop the notebook rather than silently change source data.


In [3]:

quality = au.quality_report(wide)
quality_frame = pd.DataFrame(
    [(key, value) for key, value in quality.items() if key != "dtypes" and key != "zero_activity_rows"],
    columns=["check", "value"],
)
display(quality_frame.to_string(index=False))

zeros = pd.DataFrame(
    list(quality["zero_activity_rows"].items()), columns=["transport", "zero_activity_days"]
)
display(zeros)

assert quality["rows"] == 1706 and quality["columns"] == 15
assert quality["missing_values"] == 0
assert quality["duplicate_rows"] == 0 and quality["duplicate_dates"] == 0
assert quality["gap_days"] == 0
assert quality["negative_values"] == 0
print("PASS: all Week 1 data-quality invariants hold; zero values were retained.")


'          check               value\n           rows                1706\n        columns                  15\n       date_min 2020-03-01 00:00:00\n       date_max 2024-10-31 00:00:00\n  calendar_days                1706\n       gap_days                   0\n missing_values                   0\n duplicate_rows                   0\nduplicate_dates                   0\nnegative_values                   0'

,transport,zero_activity_days
0,Subways,0
1,Buses,0
2,LIRR,0
3,Metro-North,0
4,Access-A-Ride,0
5,Bridges and Tunnels,0
6,Staten Island Railway,32


PASS: all Week 1 data-quality invariants hold; zero values were retained.



## 5. Exploratory Data Analysis

Descriptive statistics are shown per system because units differ. Histograms use small multiples with system-specific x-axes so incompatible units are never shared on one scale.


In [4]:

desc = (
    long.groupby("transport", sort=False)
    .agg(
        measure=("measure", "first"),
        unit=("unit", "first"),
        mean=("activity", "mean"),
        median=("activity", "median"),
        std=("activity", "std"),
        min=("activity", "min"),
        max=("activity", "max"),
        zero_days=("activity", lambda s: int((s == 0).sum())),
    )
    .reindex(au.SYSTEM_NAMES)
)
display(desc)

fig, axes = plt.subplots(4, 2, figsize=(15, 10))
for ax, cfg in zip(axes.ravel(), au.TRANSPORT_CONFIG):
    values = long.loc[long["transport"] == cfg["name"], "activity"]
    ax.hist(values, bins=40, color=au.SYSTEM_COLORS[cfg["name"]], alpha=0.8)
    ax.set_title(cfg["name"])
    ax.set_xlabel(cfg["unit"])
    ax.set_ylabel("Number of days")
axes.ravel()[-1].axis("off")
fig.suptitle("Distribution of daily activity by system (separate unit scales)")
fig.tight_layout(rect=[0, 0, 1, 0.97])
save_fig(fig, "01_activity_distributions.png")

anomalies = au.anomaly_summary(long)
display(anomalies)


,measure,unit,mean,median,std,min,max,zero_days
transport,,,,,,,,
Subways,Estimated Ridership,estimated riders,"2,509,054.98","2,459,607.00","1,062,183.68",198399,5498809,0
Buses,Estimated Ridership,estimated riders,"1,006,868.09","1,140,776.50","440,379.92",5498,2244515,0
LIRR,Estimated Ridership,estimated riders,"135,960.12","124,274.00","71,298.78",1903,321569,0
Metro-North,Estimated Ridership,estimated riders,"114,888.27","108,237.00","66,500.21",3281,249585,0
Access-A-Ride,Scheduled Trips,scheduled trips,"21,941.53","22,301.50","7,990.63",2506,40468,0
Bridges and Tunnels,Traffic,vehicles,"855,602.87","896,634.50","143,184.01",156759,1043802,0
Staten Island Railway,Estimated Ridership,estimated riders,"4,429.36","4,522.50","2,681.66",0,17453,32


,transport,kind,date,activity,recovery_pct,pct_of_system_mean
0,Subways,high,2020-03-04,5498809,99,219.16
1,Subways,high,2020-03-05,5496453,99,219.06
2,Subways,high,2020-03-03,5481103,98,218.45
3,Subways,low,2020-04-12,198399,8,7.91
4,Subways,low,2020-04-26,218787,9,8.72
5,Subways,low,2020-04-19,224411,9,8.94
6,Buses,high,2020-03-05,2244515,100,222.92
7,Buses,high,2020-03-03,2228608,99,221.34
8,Buses,high,2020-03-02,2209066,99,219.40
9,Buses,low,2020-04-26,5498,1,0.55



## 6. Transportation System Performance

Performance uses the source recovery percentage, which is already normalised to each system's comparable 2019 day. Relative change compares the first 12 available months with the last 12 months; volatility is the standard deviation of daily activity changes divided by each system's own mean. Both are unit-free.


In [5]:

metrics = au.system_metrics(long)
performance_columns = [
    "transport", "measure", "unit", "mean_activity", "cv_activity",
    "daily_change_volatility", "period_change_pct", "late_recovery_mean",
    "days_above_100", "late_share_above_100_pct", "sustained_100_date",
]
display(metrics[performance_columns])

ranked_change = metrics.set_index("transport")["period_change_pct"].sort_values()
ranked_vol = metrics.set_index("transport")["daily_change_volatility"].sort_values()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].barh(ranked_change.index, ranked_change.values, color="#4c78a8")
axes[0].axvline(0, color="black", lw=0.8)
axes[0].set_title("Activity change: first 12 vs last 12 months")
axes[0].set_xlabel("Change (%)")
axes[1].barh(ranked_vol.index[::-1], ranked_vol.values[::-1], color="#f58518")
axes[1].set_title("Day-to-day activity volatility")
axes[1].set_xlabel("Std. of daily change / system mean (unit-free)")
fig.tight_layout()
save_fig(fig, "02_system_performance.png")


,transport,measure,unit,mean_activity,cv_activity,daily_change_volatility,period_change_pct,late_recovery_mean,days_above_100,late_share_above_100_pct,sustained_100_date
0,Subways,Estimated Ridership,estimated riders,"2,509,054.98",0.42,0.27,162.01,71.35,8,0.82,NaT
1,Buses,Estimated Ridership,estimated riders,"1,006,868.09",0.44,0.30,106.03,59.95,4,0.27,NaT
2,LIRR,Estimated Ridership,estimated riders,"135,960.12",0.52,0.35,249.60,87.49,142,25.68,NaT
3,Metro-North,Estimated Ridership,estimated riders,"114,888.27",0.58,0.34,383.50,77.98,20,3.28,NaT
4,Access-A-Ride,Scheduled Trips,scheduled trips,"21,941.53",0.36,0.26,101.19,117.54,496,92.62,2023-05-07
5,Bridges and Tunnels,Traffic,vehicles,"855,602.87",0.17,0.09,37.14,101.07,556,56.01,2021-12-04
6,Staten Island Railway,Estimated Ridership,estimated riders,"4,429.36",0.61,0.51,140.72,50.88,19,1.37,NaT



## 7. Daily and Monthly Trends

Daily charts show 7-day means per system to reduce weekly noise while preserving zeros. Monthly recovery charts use the published percentage directly. The month-of-year heatmap indexes activity to each year's own mean, so recurring seasonal effects are visible without being dominated by the 2020-2024 recovery trend.


In [6]:

monthly = au.monthly_summary(long)
seasonality = au.seasonality_index(long)

fig, axes = plt.subplots(4, 2, figsize=(16, 11), sharex=True)
for ax, cfg in zip(axes.ravel(), au.TRANSPORT_CONFIG):
    series = long.loc[long["transport"] == cfg["name"]].sort_values("date")
    rolling = series["activity"].rolling(7, min_periods=7).mean()
    ax.plot(series["date"], rolling, color=au.SYSTEM_COLORS[cfg["name"]], lw=1.2)
    ax.set_title(cfg["name"])
    ax.set_ylabel(cfg["unit"])
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
axes.ravel()[-1].axis("off")
fig.suptitle("7-day mean daily activity by system, March 2020 - October 2024")
fig.tight_layout(rect=[0, 0, 1, 0.97])
save_fig(fig, "03_daily_activity_trends.png")

fig, ax = plt.subplots(figsize=(14, 6))
for name in au.SYSTEM_NAMES:
    sub = monthly[monthly["transport"] == name]
    ax.plot(sub["month_start"], sub["recovery_mean"], label=name, lw=1.6, color=au.SYSTEM_COLORS[name])
ax.axhline(100, color="black", ls="--", lw=1, label="Comparable 2019 day = 100%")
ax.axvline(au.METHODOLOGY_DATE, color="grey", ls=":", lw=1.5)
ax.text(au.METHODOLOGY_DATE, 5, " Feb 2023 comparison-methodology update", color="grey", fontsize=9)
ax.set_title("Monthly mean recovery by system (% of comparable 2019 day)")
ax.set_xlabel("Month")
ax.set_ylabel("Recovery (% of comparable 2019 day)")
ax.legend(ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.15))
ax.margins(x=0.01)
fig.tight_layout()
save_fig(fig, "04_monthly_recovery_trends.png")

fig, ax = plt.subplots(figsize=(12, 6))
image = ax.imshow(seasonality.values, cmap="YlGnBu", aspect="auto")
ax.set_xticks(range(seasonality.shape[1]), seasonality.columns)
ax.set_yticks(range(seasonality.shape[0]), seasonality.index)
for row in range(seasonality.shape[0]):
    for col in range(seasonality.shape[1]):
        value = seasonality.iloc[row, col]
        ax.text(col, row, f"{value:.0f}", ha="center", va="center", fontsize=8,
                color="white" if value > 112 else "black")
fig.colorbar(image, ax=ax, label="Activity index (100 = that year's mean)")
ax.set_title("Month-of-year activity pattern, indexed within each year")
ax.set_xlabel("Calendar month")
ax.set_ylabel("Transportation system")
fig.tight_layout()
save_fig(fig, "05_month_of_year_heatmap.png")



## 8. Weekday and Weekend Analysis

Day type is derived from `dim_date.day_name` via the Week 1 date dimension. Ratios are computed within each system, so different units remain separate.


In [7]:

day_type = au.day_type_summary(long)
day_pivot = day_type.pivot(index="transport", columns="day_type", values="mean_activity").reindex(au.SYSTEM_NAMES)
day_ratio = (day_pivot["Weekend"] / day_pivot["Weekday"]).rename("weekend_weekday_ratio")
display(day_type)
display(day_ratio.to_frame())

fig, ax = plt.subplots(figsize=(10, 5))
order = day_ratio.sort_values()
ax.barh(order.index, order.values, color="#55a868")
ax.axvline(1, color="black", ls="--", lw=1)
ax.set_title("Weekend / weekday mean activity by system (unit-free ratio)")
ax.set_xlabel("Weekend mean divided by weekday mean")
ax.set_ylabel("Transportation system")
fig.tight_layout()
save_fig(fig, "06_weekday_weekend_ratio.png")


,transport,day_type,mean_activity,median_activity,mean_recovery,days
0,Access-A-Ride,Weekday,"24,889.58","24,338.00",86.42,1219
1,Access-A-Ride,Weekend,"14,562.34","14,367.00",85.52,487
2,Bridges and Tunnels,Weekday,"873,232.11","909,180.00",93.69,1219
3,Bridges and Tunnels,Weekend,"811,475.45","856,286.00",92.59,487
4,Buses,Weekday,"1,152,092.23","1,261,796.00",54.68,1219
5,Buses,Weekend,"643,360.45","682,092.00",54.73,487
6,LIRR,Weekday,"159,010.72","171,765.00",53.16,1219
7,LIRR,Weekend,"78,262.62","80,874.00",74.08,487
8,Metro-North,Weekday,"132,847.23","148,590.00",48.91,1219
9,Metro-North,Weekend,"69,935.57","74,415.00",56.53,487


,weekend_weekday_ratio
transport,
Subways,0.59
Buses,0.56
LIRR,0.49
Metro-North,0.53
Access-A-Ride,0.59
Bridges and Tunnels,0.93
Staten Island Railway,0.27



## 9. COVID-19 Impact

The dataset starts on 2020-03-01, so there is no direct pre-2020 activity to compare. Impact is therefore measured with the published recovery percentage, which already encodes each day's comparison with a 2019 equivalent day. March-April 2020 is used as the early-pandemic window.


In [8]:

covid = metrics[
    ["transport", "early_recovery_mean", "early_recovery_min", "full_recovery_min",
     "full_recovery_min_date", "late_recovery_mean"]
].sort_values("early_recovery_mean")
display(covid)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
early_sorted = covid.sort_values("early_recovery_mean")
axes[0].barh(early_sorted["transport"], early_sorted["early_recovery_mean"], color="#e45756")
axes[0].axvline(100, color="black", ls="--", lw=1)
axes[0].set_title("March-April 2020 mean recovery")
axes[0].set_xlabel("Recovery (% of comparable 2019 day)")
minimum_sorted = covid.sort_values("full_recovery_min")
axes[1].barh(minimum_sorted["transport"], minimum_sorted["full_recovery_min"], color="#72b7b2")
axes[1].set_title("Lowest daily recovery in the dataset")
axes[1].set_xlabel("Recovery (% of comparable 2019 day)")
fig.tight_layout()
save_fig(fig, "07_covid_impact.png")


,transport,early_recovery_mean,early_recovery_min,full_recovery_min,full_recovery_min_date,late_recovery_mean
2,LIRR,24.97,2,2,2020-04-11,87.49
6,Staten Island Railway,27.21,4,0,2020-05-30,50.88
1,Buses,27.64,1,1,2020-03-28,59.95
3,Metro-North,27.77,3,3,2020-04-01,77.98
0,Subways,29.66,7,7,2020-04-09,71.35
4,Access-A-Ride,46.18,13,13,2020-04-26,117.54
5,Bridges and Tunnels,52.56,24,18,2021-02-01,101.07



## 10. Recovery Analysis

Recovery is the source percentage of a comparable 2019 month/day-type day. A value of 100 means the baseline was met; values above 100 are retained. "Sustained recovery" is defined as a 7-day rolling mean of at least 100 for 14 consecutive days.


In [9]:

recovery_table = metrics[
    ["transport", "late_recovery_mean", "days_above_100", "late_share_above_100_pct",
     "max_recovery", "max_recovery_date", "first_daily_100_date",
     "first_roll7_100_date", "sustained_100_date", "longest_daily_100_streak"]
]
display(recovery_table)

fig, ax = plt.subplots(figsize=(12, 5))
recovery_values = [long.loc[long["transport"] == name, "recovery_pct"] for name in au.SYSTEM_NAMES]
ax.boxplot(recovery_values, tick_labels=au.SYSTEM_NAMES, showfliers=False, patch_artist=True)
ax.axhline(100, color="black", ls="--", lw=1, label="Comparable 2019 day = 100%")
ax.set_title("Distribution of daily recovery percentages by system")
ax.set_xlabel("Transportation system")
ax.set_ylabel("Recovery (% of comparable 2019 day)")
ax.tick_params(axis="x", rotation=30)
ax.legend()
fig.tight_layout()
save_fig(fig, "08_recovery_distribution.png")

fig, axes = plt.subplots(4, 2, figsize=(16, 11), sharex=True)
for ax, cfg in zip(axes.ravel(), au.TRANSPORT_CONFIG):
    series = long.loc[long["transport"] == cfg["name"]].sort_values("date")
    rolling = series["recovery_pct"].rolling(7, min_periods=7).mean()
    ax.plot(series["date"], rolling, color=au.SYSTEM_COLORS[cfg["name"]], lw=1.2)
    ax.axhline(100, color="black", ls="--", lw=0.8)
    ax.axvline(au.METHODOLOGY_DATE, color="grey", ls=":", lw=1)
    ax.set_title(cfg["name"])
    ax.set_ylabel("Recovery (%)")
    ax.set_ylim(0, max(130, rolling.max() * 1.05))
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
axes.ravel()[-1].axis("off")
fig.suptitle("7-day mean recovery by system; dashed line = 100% baseline, dotted line = Feb 2023 methodology update")
fig.tight_layout(rect=[0, 0, 1, 0.97])
save_fig(fig, "09_recovery_paths.png")


,transport,late_recovery_mean,days_above_100,late_share_above_100_pct,max_recovery,max_recovery_date,first_daily_100_date,first_roll7_100_date,sustained_100_date,longest_daily_100_streak
0,Subways,71.35,8,0.82,143,2023-11-10,2021-11-11,NaT,NaT,1
1,Buses,59.95,4,0.27,126,2023-11-10,2020-03-05,NaT,NaT,1
2,LIRR,87.49,142,25.68,237,2023-11-10,2020-03-01,2023-06-19,NaT,4
3,Metro-North,77.98,20,3.28,193,2023-11-10,2021-11-11,NaT,NaT,2
4,Access-A-Ride,117.54,496,92.62,144,2024-05-12,2020-03-01,2020-03-07,2023-05-07,103
5,Bridges and Tunnels,101.07,556,56.01,120,2024-01-15,2020-03-05,2021-10-08,2021-12-04,17
6,Staten Island Railway,50.88,19,1.37,182,2023-11-10,2020-03-02,NaT,NaT,4



## 11. Relationships and Anomalies

Similarity uses Pearson correlation of each system's 7-day activity index (activity divided by that system's own mean). It describes co-movement, not causation. The anomaly review focuses on the SIR zero days and the one-day recovery jump on 2023-11-10.


In [10]:

correlations = au.correlation_analysis(long)
corr_matrix = correlations["pearson_smoothed"]
display(corr_matrix.round(2))

fig, ax = plt.subplots(figsize=(9, 7))
image = ax.imshow(corr_matrix.values, cmap="coolwarm", vmin=0, vmax=1)
ax.set_xticks(range(7), corr_matrix.columns, rotation=45, ha="right")
ax.set_yticks(range(7), corr_matrix.index)
for row in range(7):
    for col in range(7):
        value = corr_matrix.iloc[row, col]
        ax.text(col, row, f"{value:.2f}", ha="center", va="center", fontsize=8,
                color="white" if value > 0.7 else "black")
fig.colorbar(image, ax=ax, label="Pearson correlation")
ax.set_title("Co-movement of 7-day normalised activity (unit-free)")
fig.tight_layout()
save_fig(fig, "10_correlation_heatmap.png")

window = long[(long["date"] >= "2023-11-06") & (long["date"] <= "2023-11-14")]
pivot = window.pivot(index="date", columns="transport", values="recovery_pct")
fig, ax = plt.subplots(figsize=(13, 6))
for name in au.SYSTEM_NAMES:
    ax.plot(pivot.index, pivot[name], marker="o", lw=1.4, label=name, color=au.SYSTEM_COLORS[name])
ax.axvline(pd.Timestamp("2023-11-10"), color="black", ls="--", lw=1.2)
ax.axhline(100, color="grey", ls=":", lw=1)
ax.set_title("Recovery percentages around 2023-11-10 (one-day jump across all systems)")
ax.set_xlabel("Date")
ax.set_ylabel("Recovery (% of comparable 2019 day)")
ax.legend(ncol=4, fontsize=8)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))
fig.tight_layout()
save_fig(fig, "11_recovery_anomaly_2023_11_10.png")

sir = long[long["transport"] == "Staten Island Railway"].copy()
sir["year"] = sir["date"].dt.year
zero_by_year = sir.loc[sir["activity"] == 0].groupby("year").size()
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(zero_by_year.index.astype(str), zero_by_year.values, color="#d62728")
for x, value in zip(zero_by_year.index.astype(str), zero_by_year.values):
    ax.text(x, value + 0.5, str(value), ha="center")
ax.set_title("Staten Island Railway zero-activity days by year")
ax.set_xlabel("Year")
ax.set_ylabel("Days with activity = 0")
ax.set_ylim(0, zero_by_year.max() * 1.15)
fig.tight_layout()
save_fig(fig, "12_sir_zero_activity.png")
display(zero_by_year.to_frame("zero_activity_days"))


transport,Subways,Buses,LIRR,Metro-North,Access-A-Ride,Bridges and Tunnels,Staten Island Railway
transport,,,,,,,
Subways,1.00,0.84,0.96,0.95,0.89,0.82,0.96
Buses,0.84,1.00,0.72,0.68,0.70,0.77,0.82
LIRR,0.96,0.72,1.00,0.98,0.94,0.80,0.91
Metro-North,0.95,0.68,0.98,1.00,0.89,0.73,0.89
Access-A-Ride,0.89,0.70,0.94,0.89,1.00,0.78,0.84
Bridges and Tunnels,0.82,0.77,0.80,0.73,0.78,1.00,0.75
Staten Island Railway,0.96,0.82,0.91,0.89,0.84,0.75,1.00


,zero_activity_days
year,
2020,3
2021,29



## 12. Answers to the 12 Analytical Questions

### A. Transportation System Performance

**Q1. How does ridership performance differ across the available transportation systems?**  
Why it matters: prioritisation needs a fair comparison. Columns: `recovery_percentage` (unit-free), `activity_value` per system. Method: rank systems by mean recovery in the last 12 months; do not total activity across units.

**Q2. Which systems have the highest and lowest relative changes over the analysed period?**  
Why it matters: identifies where activity shifted most. Columns: `activity_value`, `date_key`. Method: mean activity in the first 12 available months versus the last 12 months, percentage change within each system.

**Q3. Which systems show the greatest day-to-day volatility?**  
Why it matters: volatility affects service planning. Columns: `activity_value`. Method: standard deviation of daily changes divided by each system's own mean, plus coefficient of variation.


In [11]:

indexed = metrics.set_index("transport")
print("Q1. Last-12-month mean recovery, ranked (percent of comparable 2019 day):")
for name, value in indexed["late_recovery_mean"].sort_values(ascending=False).items():
    print(f"  {name}: {value:.1f}%")

print("\nQ2. First-12 to last-12 month activity change, ranked:")
for name, value in indexed["period_change_pct"].sort_values(ascending=False).items():
    print(f"  {name}: {value:+.1f}%")

print("\nQ3. Day-to-day volatility (std of daily change / own mean), ranked:")
for name, value in indexed["daily_change_volatility"].sort_values(ascending=False).items():
    print(f"  {name}: {value:.3f}; coefficient of variation = {indexed.loc[name, 'cv_activity']:.2f}")
print("\nRelated charts: 02_system_performance.png, 08_recovery_distribution.png")


Q1. Last-12-month mean recovery, ranked (percent of comparable 2019 day):
  Access-A-Ride: 117.5%
  Bridges and Tunnels: 101.1%
  LIRR: 87.5%
  Metro-North: 78.0%
  Subways: 71.3%
  Buses: 59.9%
  Staten Island Railway: 50.9%

Q2. First-12 to last-12 month activity change, ranked:
  Metro-North: +383.5%
  LIRR: +249.6%
  Subways: +162.0%
  Staten Island Railway: +140.7%
  Buses: +106.0%
  Access-A-Ride: +101.2%
  Bridges and Tunnels: +37.1%

Q3. Day-to-day volatility (std of daily change / own mean), ranked:
  Staten Island Railway: 0.514; coefficient of variation = 0.61
  LIRR: 0.348; coefficient of variation = 0.52
  Metro-North: 0.340; coefficient of variation = 0.58
  Buses: 0.301; coefficient of variation = 0.44
  Subways: 0.267; coefficient of variation = 0.42
  Access-A-Ride: 0.260; coefficient of variation = 0.36
  Bridges and Tunnels: 0.088; coefficient of variation = 0.17

Related charts: 02_system_performance.png, 08_recovery_distribution.png



### B. Time-Based Analysis

**Q4. How does daily ridership change over time for each system?**  
Why it matters: shows the shape of recovery. Columns: `date_key`, `activity_value`. Method: 7-day rolling mean per system.

**Q5. What are the monthly trends and which months are consistently higher or lower?**  
Why it matters: supports scheduling and budget timing. Columns: `date_key`, `activity_value`. Method: monthly means and an activity index within each year to reduce trend confounding.

**Q6. Are there weekday/weekend differences?**  
Why it matters: service plans differ by day type. Columns: `dim_date.day_name`, `activity_value`. Method: mean activity by day type and weekend/weekday ratio per system.

**Q7. What seasonal patterns or unusual peaks and drops occur?**  
Why it matters: distinguishes routine seasonality from anomalies. Columns: `date_key`, `activity_value`, `recovery_pct`. Method: month-of-year index, ranked daily extremes, and event-date review.


In [12]:

print("Q4. Monthly activity trough and peak by system:")
for name in au.SYSTEM_NAMES:
    sub = monthly[monthly["transport"] == name]
    low = sub.loc[sub["activity_mean"].idxmin()]
    high = sub.loc[sub["activity_mean"].idxmax()]
    print(f"  {name}: trough {low['month']} = {low['activity_mean']:,.0f} {au.SYSTEM_UNITS[name]}; "
          f"peak {high['month']} = {high['activity_mean']:,.0f} {au.SYSTEM_UNITS[name]}")

print("\nQ5. Month-of-year activity index (100 = that year's mean), selected extremes:")
for name in au.SYSTEM_NAMES:
    row = seasonality.loc[name]
    print(f"  {name}: high {row.idxmax()} = {row.max():.0f}; low {row.idxmin()} = {row.min():.0f}")

print("\nQ6. Weekend / weekday mean activity ratio:")
for name, value in day_ratio.sort_values(ascending=False).items():
    print(f"  {name}: {value:.2f}")

print("\nQ7. Unusual daily observations (largest negative relative to system mean):")
top_drop = (anomalies[anomalies["kind"] == "low"]
            .sort_values("pct_of_system_mean")
            .groupby("transport", sort=False)
            .head(1))
for _, row in top_drop.iterrows():
    print(f"  {row['transport']}: {row['date'].date()} = {row['activity']:,.0f} "
          f"({row['pct_of_system_mean']:.1f}% of its mean), recovery {row['recovery_pct']}%")
print("\nRelated charts: 03_daily_activity_trends.png, 04_monthly_recovery_trends.png, "
      "05_month_of_year_heatmap.png, 06_weekday_weekend_ratio.png")


Q4. Monthly activity trough and peak by system:
  Subways: trough 2020-04 = 391,016 estimated riders; peak 2024-10 = 3,658,850 estimated riders
  Buses: trough 2020-05 = 13,249 estimated riders; peak 2024-10 = 1,294,158 estimated riders
  LIRR: trough 2020-04 = 7,244 estimated riders; peak 2024-10 = 229,105 estimated riders
  Metro-North: trough 2020-04 = 8,145 estimated riders; peak 2024-10 = 204,956 estimated riders
  Access-A-Ride: trough 2020-04 = 6,062 scheduled trips; peak 2024-10 = 33,769 scheduled trips
  Bridges and Tunnels: trough 2020-04 = 330,540 vehicles; peak 2024-06 = 982,010 vehicles
  Staten Island Railway: trough 2020-04 = 656 estimated riders; peak 2024-10 = 6,715 estimated riders

Q5. Month-of-year activity index (100 = that year's mean), selected extremes:
  Subways: high Oct = 118; low Jan = 80
  Buses: high Oct = 129; low Jul = 78
  LIRR: high Nov = 121; low Jan = 72
  Metro-North: high Mar = 134; low Jan = 68
  Access-A-Ride: high Oct = 112; low Jan = 84
  Bridg


### C. COVID-19 Impact and Recovery

**Q8. How significantly did COVID-19 affect ridership across systems?**  
Why it matters: quantifies the shock. Columns: `recovery_percentage`, `activity_value`, `date_key`. Method: March-April 2020 mean and minimum recovery. Limitation: no pre-2020 activity exists in this dataset.

**Q9. How did each system recover relative to its baseline?**  
Why it matters: measures return to normal. Columns: `recovery_percentage`. Method: distribution and last-12-month mean, counts of days above 100, and investigation of extreme values.

**Q10. When did each system reach or exceed its baseline?**  
Why it matters: marks operational normalisation. Columns: `recovery_percentage`, `date_key`. Method: first date with a 7-day rolling mean >= 100 sustained for 14 consecutive days; isolated daily crossings are reported separately.


In [13]:

print("Q8. March-April 2020 mean recovery and lowest daily recovery:")
for _, row in covid.iterrows():
    print(f"  {row['transport']}: early mean {row['early_recovery_mean']:.1f}%, "
          f"window minimum {row['early_recovery_min']:.0f}%, "
          f"overall minimum {row['full_recovery_min']:.0f}% on {row['full_recovery_min_date'].date()}")

print("\nQ9. Recovery status by system:")
for _, row in recovery_table.iterrows():
    print(f"  {row['transport']}: last-12m mean {row['late_recovery_mean']:.1f}%, "
          f"{int(row['days_above_100'])} days >100, "
          f"max {row['max_recovery']:.0f}% on {row['max_recovery_date'].date()}")

print("\nQ10. First sustained 7-day mean >=100 for 14 consecutive days:")
for _, row in recovery_table.iterrows():
    print(f"  {row['transport']}: sustained {fmt_date(row['sustained_100_date'])}; "
          f"first 7-day crossing {fmt_date(row['first_roll7_100_date'])}; "
          f"longest daily streak {int(row['longest_daily_100_streak'])} day(s)")
print("\nRelated charts: 07_covid_impact.png, 08_recovery_distribution.png, 09_recovery_paths.png")


Q8. March-April 2020 mean recovery and lowest daily recovery:
  LIRR: early mean 25.0%, window minimum 2%, overall minimum 2% on 2020-04-11
  Staten Island Railway: early mean 27.2%, window minimum 4%, overall minimum 0% on 2020-05-30
  Buses: early mean 27.6%, window minimum 1%, overall minimum 1% on 2020-03-28
  Metro-North: early mean 27.8%, window minimum 3%, overall minimum 3% on 2020-04-01
  Subways: early mean 29.7%, window minimum 7%, overall minimum 7% on 2020-04-09
  Access-A-Ride: early mean 46.2%, window minimum 13%, overall minimum 13% on 2020-04-26
  Bridges and Tunnels: early mean 52.6%, window minimum 24%, overall minimum 18% on 2021-02-01

Q9. Recovery status by system:
  Subways: last-12m mean 71.3%, 8 days >100, max 143% on 2023-11-10
  Buses: last-12m mean 59.9%, 4 days >100, max 126% on 2023-11-10
  LIRR: last-12m mean 87.5%, 142 days >100, max 237% on 2023-11-10
  Metro-North: last-12m mean 78.0%, 20 days >100, max 193% on 2023-11-10
  Access-A-Ride: last-12m mean


### D. Relationships and Operational Insights

**Q11. Which systems show similar movement patterns?**  
Why it matters: shared signals can support network-level monitoring. Columns: `activity_value` per system. Method: Pearson correlation of 7-day activity indices; association only.

**Q12. What unusual patterns or operational signals deserve follow-up?**  
Why it matters: separates data-quality signals from demand signals. Columns: `activity_value`, `recovery_pct`, `date_key`. Method: ranked extremes, SIR zero counts, and the 2023-11-10 recovery jump.


In [14]:

pairs = [(("LIRR", "Metro-North")), (("Subways", "LIRR")), (("Subways", "Staten Island Railway")),
         (("LIRR", "Access-A-Ride")), (("Subways", "Buses")), (("Subways", "Bridges and Tunnels"))]
print("Q11. Correlation of 7-day normalised activity (unit-free; not causal):")
for left, right in pairs:
    print(f"  {left} vs {right}: {corr_matrix.loc[left, right]:.2f}")
b_t = corr_matrix.loc["Bridges and Tunnels"].drop("Bridges and Tunnels")
print(f"  Bridges and Tunnels: lowest correlation with {b_t.idxmin()} ({b_t.min():.2f})")

print("\nQ12. Operational signals:")
sir_zeros = int(metrics.set_index("transport").loc["Staten Island Railway", "zero_activity_days"])
print(f"  SIR zero-activity days retained: {sir_zeros}, all on weekends (3 in 2020, 29 in 2021).")
print("  On 2023-11-10 recovery jumped sharply in every system while activity did not;")
print("  this points to the comparison baseline/denominator rather than a demand surge.")
print("  February 2023 methodology update affects recovery percentages, not activity estimates.")
print("\nRelated charts: 10_correlation_heatmap.png, 11_recovery_anomaly_2023_11_10.png, 12_sir_zero_activity.png")


Q11. Correlation of 7-day normalised activity (unit-free; not causal):
  LIRR vs Metro-North: 0.98
  Subways vs LIRR: 0.96
  Subways vs Staten Island Railway: 0.96
  LIRR vs Access-A-Ride: 0.94
  Subways vs Buses: 0.84
  Subways vs Bridges and Tunnels: 0.82
  Bridges and Tunnels: lowest correlation with Metro-North (0.73)

Q12. Operational signals:
  SIR zero-activity days retained: 32, all on weekends (3 in 2020, 29 in 2021).
  On 2023-11-10 recovery jumped sharply in every system while activity did not;
  this points to the comparison baseline/denominator rather than a demand surge.
  February 2023 methodology update affects recovery percentages, not activity estimates.

Related charts: 10_correlation_heatmap.png, 11_recovery_anomaly_2023_11_10.png, 12_sir_zero_activity.png



## 13. Key Business Insights

**Observed facts**

- In the last 12 months, Access-A-Ride averaged 117.5% and Bridges and Tunnels 101.1% of their comparable 2019 day; Staten Island Railway averaged 50.9% and Buses 60.0%.
- First-to-last 12-month activity changes were largest for Metro-North (+383.5%) and LIRR (+249.6%), and smallest for Bridges and Tunnels (+37.1%). The first window is the COVID-affected year, so these are recovery gaps, not normal growth rates.
- Bridges and Tunnels had the steadiest daily activity (change volatility 0.09 of its mean); Staten Island Railway the most volatile (0.51).
- Weekend/weekday activity ratios range from 0.93 for Bridges and Tunnels to 0.27 for Staten Island Railway, which also has 32 zero-activity weekend days.
- 2023-11-10 shows a one-day recovery jump in all seven systems without a matching activity jump.

**Analytical interpretations**

- Systems with high late-period recovery (Access-A-Ride, Bridges and Tunnels) have largely closed or passed their 2019 comparison gap, but their units differ (scheduled trips versus vehicles), so their absolute activity must not be added together.
- The 2023-11-10 pattern is more consistent with a comparison-baseline artefact than with a demand event, because the activity values on that date are close to neighbouring days.
- Strong correlations among Subways, LIRR, Metro-North, Access-A-Ride and SIR describe shared weekly/seasonal movement; they do not establish a causal link.

**Recommendations**

1. Validate the 2019 comparison denominator for 2023-11-10 before using that day in KPI reporting.
2. Investigate the 32 SIR zero weekends with the service operator to distinguish outages from reporting gaps; keep them in the data.
3. Treat recovery percentages before and after February 2023 as methodologically distinct when reporting trends, and document the refreshed baseline.
4. Report system KPIs separately by unit family; use recovery percentage or indices for cross-system comparisons.

**Limitations**

- No direct pre-2020 activity values are present, so COVID impact relies on the published recovery percentage.
- Activity estimates for some modes are modelled or preliminary; source documentation warns that recent days may be revised.
- Correlation is not causation, and the dataset contains no service, fare, weather or event variables to explain causes.


In [15]:

metrics.to_csv(REPORT_DIR / "system_metrics.csv", index=False)
corr_matrix.to_csv(REPORT_DIR / "correlation_matrix.csv")
print(f"Saved reproducible metric exports to {REPORT_DIR.relative_to(PROJECT_ROOT).as_posix()}")


Saved reproducible metric exports to WEEK TWO/reports



## 14. Final Conclusions

Week 2 establishes that the seven systems have materially different recovery paths and different units. Cross-system conclusions are therefore based on the published recovery percentage, within-system indices, ratios and correlations. The COVID period produced very low March-April 2020 recovery across all systems, with the deepest minima in bus and rail modes. By the last 12 months, Access-A-Ride and Bridges and Tunnels had reached or exceeded their comparison baseline, while subway, bus and Staten Island Railway recovery remained materially below 100. Day-to-day volatility is highest for Staten Island Railway and lowest for Bridges and Tunnels. Two anomalies require follow-up before decisions: the SIR zero weekends and the one-day recovery jump on 2023-11-10. All findings are reproducible from the Week 1 cleaned CSV and the project-relative scripts in `WEEK TWO/`.
